# Data Cleaning — E-commerce Discount Analysis

In this notebook I am cleaning the raw data collected from 
Flipkart and Shopsy. The main issues I found were:
- Some products didn't have MRP listed
- Star ratings were missing for many products  
- Some headphone product names had extra text like prices 
  and ratings stuck to them from the scraper
- A few rows had ratings as product names (scraping artifact)

I will fix all of these step by step below.

In [41]:
import pandas as pd
import numpy as np
import re
import os

print("All imports done!")

All imports done!


In [40]:
df = pd.read_csv("master_raw.csv")

print(f"Total rows loaded : {len(df)}")
print(f"Total columns     : {len(df.columns)}")
print(f"Columns           : {list(df.columns)}")
df.head()

Total rows loaded : 440
Total columns     : 12
Columns           : ['product_name', 'selling_price_inr', 'mrp_inr', 'discount_as_listed', 'discount_calculated', 'star_rating', 'specs', 'category', 'platform', 'source_url', 'scraped_at', 'num_reviews']


,product_name,selling_price_inr,mrp_inr,discount_as_listed,discount_calculated,star_rating,specs,category,platform,source_url,scraped_at,num_reviews
0,"vivo T5x 5G (Star Silver, 128 GB)",22999,28999.0,20.0,20.69,4.5,6 GB RAM | 128 GB ROM17.17 cm (6.76 inch) Disp...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:10,NaN
1,"vivo T5x 5G (Cyber Green, 128 GB)",22999,28999.0,20.0,20.69,4.5,6 GB RAM | 128 GB ROM17.17 cm (6.76 inch) Disp...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:10,NaN
2,"MOTOROLA g06 power (Pantone tapestry, 64 GB)",9999,NaN,NaN,NaN,4.3,4 GB RAM | 64 GB ROM | Expandable Upto 1 TB17....,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:10,NaN
3,"realme P4 Lite 5G (Mosaic Blue, 64 GB)",13499,19999.0,32.0,32.50,4.4,4 GB RAM | 64 GB ROM17.27 cm (6.8 inches) HD+ ...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:10,NaN
4,"vivo T5x 5G (Cyber Green, 256 GB)",26999,33999.0,20.0,20.59,4.5,8 GB RAM | 256 GB ROM17.17 cm (6.76 inch) Disp...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:10,NaN


In [29]:
print("=== BASIC INFO ===")
print(df.info())

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DUPLICATE ROWS ===")
print(f"Duplicate rows: {df.duplicated().sum()}")

print("\n=== SAMPLE DATA ===")
df.sample(5)

=== BASIC INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_name         440 non-null    object 
 1   selling_price_inr    440 non-null    int64  
 2   mrp_inr              409 non-null    float64
 3   discount_as_listed   371 non-null    float64
 4   discount_calculated  409 non-null    float64
 5   star_rating          225 non-null    float64
 6   specs                187 non-null    object 
 7   category             440 non-null    object 
 8   platform             440 non-null    object 
 9   source_url           440 non-null    object 
 10  scraped_at           440 non-null    object 
 11  num_reviews          208 non-null    object 
dtypes: float64(4), int64(1), object(7)
memory usage: 41.4+ KB
None

=== MISSING VALUES ===
product_name             0
selling_price_inr        0
mrp_inr                 31
dis

,product_name,selling_price_inr,mrp_inr,discount_as_listed,discount_calculated,star_rating,specs,category,platform,source_url,scraped_at,num_reviews
287,"realme P4 5G (Forge Red, 128 GB)",20989,20999.0,NaN,0.05,NaN,NaN,mobiles,Shopsy,https://www.shopsy.in/search?q=smartphones&page=3,2026-04-21 17:20:43,"(23,080)"
33,"realme P4 Lite (Sea Blue, 64 GB)",9999,12999.0,23.0,23.08,4.4,4 GB RAM | 64 GB ROM17.14 cm (6.75 inch) HD+ D...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:26,NaN
306,"vivo T4x 5G (Glacial Teal, 256 GB)",20989,20999.0,NaN,0.05,NaN,NaN,mobiles,Shopsy,https://www.shopsy.in/search?q=smartphones&page=4,2026-04-21 17:20:48,"(1,68,924)"
96,"MOTOROLA g05 (Plum Red, 64 GB)",7299,9999.0,27.0,27.00,4.2,4 GB RAM | 64 GB ROM | Expandable Upto 1 TB16....,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:32:01,NaN
141,DELL Inspiron 15 Microsoft Office Home 2024 In...,54990,63416.0,13.0,13.29,4.2,Intel Core i5 Processor (13th Gen)8 GB DDR4 RA...,laptops,Flipkart,https://www.flipkart.com/search?q=laptops&page=2,2026-04-21 15:32:23,NaN


## problems i faced during cleaning

- the headphone product names had prices and ratings 
  stuck to them which took me a while to fix
- some rows had just a rating number like "3.6" as 
  the product name which i had to remove
- MRP was missing for 31 products because some listings 
  on flipkart dont show original price

In [30]:
before = len(df)

df.drop_duplicates(
    subset=["product_name", "selling_price_inr", "category", "platform"],
    inplace=True
)
df.reset_index(drop=True, inplace=True)

after = len(df)

print(f"Rows before : {before}")
print(f"Rows after  : {after}")
print(f"Removed     : {before - after} duplicates")

Rows before : 440
Rows after  : 440
Removed     : 0 duplicates


In [31]:
df["selling_price_inr"]  = pd.to_numeric(df["selling_price_inr"], errors="coerce")
df["mrp_inr"]            = pd.to_numeric(df["mrp_inr"], errors="coerce")
df["discount_calculated"]= pd.to_numeric(df["discount_calculated"], errors="coerce")
df["discount_as_listed"] = pd.to_numeric(df["discount_as_listed"], errors="coerce")
df["star_rating"]        = pd.to_numeric(df["star_rating"], errors="coerce")

df["scraped_at"] = pd.to_datetime(df["scraped_at"], errors="coerce")

print("Data types after fixing:")
print(df.dtypes)

Data types after fixing:
product_name                   object
selling_price_inr               int64
mrp_inr                       float64
discount_as_listed            float64
discount_calculated           float64
star_rating                   float64
specs                          object
category                       object
platform                       object
source_url                     object
scraped_at             datetime64[ns]
num_reviews                    object
dtype: object


In [32]:
print("Missing values BEFORE:")
print(df.isnull().sum())

df["mrp_inr"] = df["mrp_inr"].fillna(df["selling_price_inr"])

mask = df["discount_calculated"].isnull()
df.loc[mask, "discount_calculated"] = (
    (df.loc[mask, "mrp_inr"] - df.loc[mask, "selling_price_inr"])
    / df.loc[mask, "mrp_inr"] * 100
).round(2)

df["discount_as_listed"] = df["discount_as_listed"].fillna(df["discount_calculated"])

df["star_rating"] = df.groupby("category")["star_rating"].transform(
    lambda x: x.fillna(x.median())
)

print("\nMissing values AFTER:")
print(df.isnull().sum())

Missing values BEFORE:
product_name             0
selling_price_inr        0
mrp_inr                 31
discount_as_listed      69
discount_calculated     31
star_rating            215
specs                  253
category                 0
platform                 0
source_url               0
scraped_at               0
num_reviews            232
dtype: int64

Missing values AFTER:
product_name             0
selling_price_inr        0
mrp_inr                  0
discount_as_listed       0
discount_calculated      0
star_rating              0
specs                  253
category                 0
platform                 0
source_url               0
scraped_at               0
num_reviews            232
dtype: int64


In [33]:
def clean_product_name(name):
    name = str(name)
    
    if "₹" in name:
        name = name[:name.index("₹")].strip()
    
    if "% off" in name or "Add to Compare" in name:
        name = re.split(r'\d+\.\d+\(|\d+\(', name)[0].strip()
    
    name = re.sub(r'\d+\.\d+\(\d+[\d,]*\)$', '', name).strip()
    
    name = name.rstrip(".").rstrip("…").strip()
    
    return name

df["product_name"] = df["product_name"].apply(clean_product_name)

print(f"Product names cleaned!")
print(f"Messy names remaining: {df['product_name'].str.contains('₹|% off', regex=True).sum()}")
print("\nSample of cleaned names:")
for name in df["product_name"].head(10):
    print(" →", name[:70])

Product names cleaned!
Messy names remaining: 0

Sample of cleaned names:
 → vivo T5x 5G (Star Silver, 128 GB)
 → vivo T5x 5G (Cyber Green, 128 GB)
 → MOTOROLA g06 power (Pantone tapestry, 64 GB)
 → realme P4 Lite 5G (Mosaic Blue, 64 GB)
 → vivo T5x 5G (Cyber Green, 256 GB)
 → Ai+ Pulse 1 (Sparkle Red, 64 GB)
 → realme P4 Lite 5G (Mosaic Green, 64 GB)
 → realme P4x 5G (Matte Silver, 128 GB)
 → Ai+ Pulse 1 (Blue, 64 GB)
 → Ai+ Pulse 2 (Purple, 64 GB)


In [34]:
before = len(df)

df = df[df["selling_price_inr"] > 0]

df = df[df["selling_price_inr"] <= df["mrp_inr"]]

df = df[(df["discount_calculated"] >= 0) & (df["discount_calculated"] <= 100)]

df.reset_index(drop=True, inplace=True)
after = len(df)

print(f"Rows before : {before}")
print(f"Rows after  : {after}")
print(f"Removed     : {before - after} bad rows")

Rows before : 440
Rows after  : 440
Removed     : 0 bad rows


In [35]:
def price_band(price):
    if price <= 10000:
        return "Budget"
    elif price <= 30000:
        return "Mid Range"
    else:
        return "Premium"

def discount_band(discount):
    if discount <= 10:
        return "Low (0-10%)"
    elif discount <= 30:
        return "Medium (10-30%)"
    else:
        return "High (30%+)"

df["price_band"]    = df["selling_price_inr"].apply(price_band)
df["discount_band"] = df["discount_calculated"].apply(discount_band)
df["savings_inr"]   = (df["mrp_inr"] - df["selling_price_inr"]).round(2)

print("New columns added: price_band, discount_band, savings_inr")
print("\nPrice band distribution:")
print(df["price_band"].value_counts())
print("\nDiscount band distribution:")
print(df["discount_band"].value_counts())

New columns added: price_band, discount_band, savings_inr

Price band distribution:
Mid Range    174
Premium      140
Budget       126
Name: price_band, dtype: int64

Discount band distribution:
Medium (10-30%)    166
Low (0-10%)        145
High (30%+)        129
Name: discount_band, dtype: int64


In [36]:
print("=== FINAL CLEANED DATASET ===")
print(f"Total rows    : {len(df)}")
print(f"Total columns : {len(df.columns)}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nPlatforms:\n{df['platform'].value_counts()}")
print(f"\nCategories:\n{df['category'].value_counts()}")
print(f"\nDiscount stats:")
print(df["discount_calculated"].describe())

df.to_csv("cleaned_data.csv", index=False, encoding="utf-8-sig")
print(f"\nCleaned data saved to: cleaned_data.csv ✅")

=== FINAL CLEANED DATASET ===
Total rows    : 440
Total columns : 15

Columns: ['product_name', 'selling_price_inr', 'mrp_inr', 'discount_as_listed', 'discount_calculated', 'star_rating', 'specs', 'category', 'platform', 'source_url', 'scraped_at', 'num_reviews', 'price_band', 'discount_band', 'savings_inr']

Platforms:
Flipkart    227
Shopsy      213
Name: platform, dtype: int64

Categories:
mobiles       206
laptops       140
headphones     94
Name: category, dtype: int64

Discount stats:
count    440.000000
mean      26.202386
std       26.583219
min        0.000000
25%        7.537500
50%       18.360000
75%       34.375000
max       90.600000
Name: discount_calculated, dtype: float64

Cleaned data saved to: cleaned_data.csv ✅


In [37]:
df.head(10)

,product_name,selling_price_inr,mrp_inr,discount_as_listed,discount_calculated,star_rating,specs,category,platform,source_url,scraped_at,num_reviews,price_band,discount_band,savings_inr
0,"vivo T5x 5G (Star Silver, 128 GB)",22999,28999.0,20.0,20.69,4.5,6 GB RAM | 128 GB ROM17.17 cm (6.76 inch) Disp...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:10,NaN,Mid Range,Medium (10-30%),6000.0
1,"vivo T5x 5G (Cyber Green, 128 GB)",22999,28999.0,20.0,20.69,4.5,6 GB RAM | 128 GB ROM17.17 cm (6.76 inch) Disp...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:10,NaN,Mid Range,Medium (10-30%),6000.0
2,"MOTOROLA g06 power (Pantone tapestry, 64 GB)",9999,9999.0,0.0,0.00,4.3,4 GB RAM | 64 GB ROM | Expandable Upto 1 TB17....,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:10,NaN,Budget,Low (0-10%),0.0
3,"realme P4 Lite 5G (Mosaic Blue, 64 GB)",13499,19999.0,32.0,32.50,4.4,4 GB RAM | 64 GB ROM17.27 cm (6.8 inches) HD+ ...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:10,NaN,Mid Range,High (30%+),6500.0
4,"vivo T5x 5G (Cyber Green, 256 GB)",26999,33999.0,20.0,20.59,4.5,8 GB RAM | 256 GB ROM17.17 cm (6.76 inch) Disp...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:10,NaN,Mid Range,Medium (10-30%),7000.0
5,"Ai+ Pulse 1 (Sparkle Red, 64 GB)",7999,7999.0,0.0,0.00,4.3,4 GB RAM | 64 GB ROM | Expandable Upto 1 TB17....,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:10,NaN,Budget,Low (0-10%),0.0
6,"realme P4 Lite 5G (Mosaic Green, 64 GB)",13499,19999.0,32.0,32.50,4.4,4 GB RAM | 64 GB ROM17.27 cm (6.8 inches) HD+ ...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:10,NaN,Mid Range,High (30%+),6500.0
7,"realme P4x 5G (Matte Silver, 128 GB)",16499,17999.0,8.0,8.33,4.4,6 GB RAM | 128 GB ROM17.07 cm (6.72 inch) Full...,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:10,NaN,Mid Range,Low (0-10%),1500.0
8,"Ai+ Pulse 1 (Blue, 64 GB)",7999,7999.0,0.0,0.00,4.3,4 GB RAM | 64 GB ROM | Expandable Upto 1 TB17....,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:10,NaN,Budget,Low (0-10%),0.0
9,"Ai+ Pulse 2 (Purple, 64 GB)",8499,10999.0,22.0,22.73,4.6,4 GB RAM | 64 GB ROM | Expandable Upto 1 TB17....,mobiles,Flipkart,https://www.flipkart.com/search?q=smartphones&...,2026-04-21 15:31:10,NaN,Budget,Medium (10-30%),2500.0


In [38]:
df = df[~df["product_name"].str.match(r'^\d+\.?\d*$', na=False)]
df.reset_index(drop=True, inplace=True)

print(f"Rows after removing invalid names: {len(df)}")

df.to_csv("cleaned_data.csv", index=False, encoding="utf-8-sig")
print("Saved!")

Rows after removing invalid names: 386
Saved!
